# Занятие 5. Кастомная трансформация и объединение данных

> **Цель занятия:** научиться менять структуру таблицы (добавлять/удалять столбцы и строки),
> применять свои функции к данным через `.apply()` и `lambda`, а также **склеивать
> несколько таблиц** в одну через `concat` и `merge`.

**Что будет:**
1. **Работа со столбцами и строками** — добавление, удаление, переименование;
2. **`.apply()` и `lambda`** — как применить свою функцию ко всей колонке или строке;
3. **`.str` accessor** — быстрая работа с текстовыми столбцами;
4. **`pd.concat()`** — склеивание таблиц «стопкой» или «в ширину»;
5. **`pd.merge()`** — соединение таблиц по общему ключу и типы соединений
   (`inner`, `left`, `right`, `outer`).


In [105]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

## Загружаем данные про ЕГЭ

Тот же датасет — 1000 учеников, 16 столбцов.

In [106]:
df = pd.read_csv('data/ege_students.csv', sep=';')
print('Размер таблицы:', df.shape)
df.head()

Размер таблицы: (1000, 16)


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


---
## Часть 1. Работа со столбцами и строками

Прежде чем изменять данные, полезно понимать, **как** менять структуру таблицы:
добавлять новые столбцы, удалять ненужные, переименовывать. Это делают почти в каждом
анализе — сразу после загрузки данных.

### 1.1. Добавляем новый столбец

Самый простой способ — просто **присвоить** значение новому столбцу:

In [107]:
df_work = df.copy()   # работаем с копией, чтобы не портить оригинал

# новый столбец: во сколько ученик готовится в день
df_work['часов_в_день'] = df_work['часов_подготовки_в_неделю'] / 7
df_work[['имя', 'часов_подготовки_в_неделю', 'часов_в_день']].head()

,имя,часов_подготовки_в_неделю,часов_в_день
0,Роман,5.4,0.771429
1,Арина,9.1,1.300000
2,Дарья,11.3,1.614286
3,Владимир,6.2,0.885714
4,Дмитрий,6.5,0.928571


**Что произошло:**
- слева от `=` мы написали `df_work['часов_в_день']` — Pandas понял: «нужно создать новый столбец с таким именем»;
- справа — формула из уже существующего столбца.

Можно и просто присвоить **одно значение всем строкам** — Pandas сам «растянет» его:

In [108]:
# добавим столбец-константу
df_work['источник_данных'] = 'ЕГЭ_2026'
df_work.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,часов_в_день,источник_данных
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да,0.771429,ЕГЭ_2026
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да,1.300000,ЕГЭ_2026
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да,1.614286,ЕГЭ_2026
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да,0.885714,ЕГЭ_2026
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да,0.928571,ЕГЭ_2026


### 1.2. Удаляем столбцы — `.drop()`

Метод **`.drop()`** удаляет колонки. Важно указать `axis=1` — это значит «столбец»
(а `axis=0` было бы «строка»).

In [109]:
# удалим временный столбец
df_work = df_work.drop('источник_данных', axis=1)
df_work.head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,часов_в_день
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да,0.771429
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да,1.300000
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да,1.614286
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да,0.885714
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да,0.928571


Можно удалить **несколько столбцов сразу**, передав список:

In [110]:
# пример: удалим сразу два столбца
tmp = df_work.drop(['часов_в_день', 'ученик_id'], axis=1)
tmp.head()

,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


> **Совет:** `.drop()` **не меняет** исходную таблицу, а возвращает **новую**.
> Поэтому обычно пишут `df = df.drop(...)`, чтобы сохранить результат.
> Если очень хочется поменять «на месте» — есть параметр `inplace=True`, но лучше
> привыкать к явной перезаписи — так меньше сюрпризов.

### 1.3. Переименовываем столбцы — `.rename()`

Передаём **словарь**: `{старое_имя: новое_имя}`.

In [111]:
# переименуем несколько столбцов
df_work = df_work.rename(columns={
    'часов_подготовки_в_неделю': 'часов_неделя',
    'кол_во_пробников': 'пробников',
})
print('Столбцы теперь:', list(df_work.columns))

Столбцы теперь: ['ученик_id', 'имя', 'пол', 'возраст', 'город', 'тип_школы', 'любимый_предмет', 'часов_неделя', 'репетитор', 'пробников', 'пропусков_уроков', 'мотивация_1_10', 'часов_сна', 'средний_балл_года', 'балл_егэ', 'сдал', 'часов_в_день']


### 1.4. Добавляем и удаляем строки

Строки добавляют реже — обычно данные приходят пачкой, и их **склеивают** через
`pd.concat()` (это дальше, в части 4). Здесь только базовые приёмы.

**Удалить строку** по её индексу:

In [112]:
# удалим строку с меткой 0 (первого ученика)
df_no_first = df_work.drop(0, axis=0)   # 0 - номер строки, axis=0 — строка
print('Было строк:', len(df_work))
print('Стало:', len(df_no_first))
df_no_first.head()

Было строк: 1000
Стало: 999


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_неделя,репетитор,пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,часов_в_день
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да,1.300000
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да,1.614286
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да,0.885714
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да,0.928571
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да,1.300000


**Удалить несколько строк** — передаём список меток:

In [113]:
df_short = df_work.drop([0, 1, 2, 3, 4], axis=0)   # удалим первые 5
df_short

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_неделя,репетитор,пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,часов_в_день
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да,1.300000
6,7,Роман,М,17,Самара,Лицей,Физика,16.6,нет,8,2,7.0,6.8,3.9,90,да,2.371429
7,8,Дарья,Ж,17,Нижний Новгород,Обычная,Биология,0.0,нет,7,12,4.0,8.6,2.5,48,да,0.000000
8,9,Тимофей,М,16,Санкт-Петербург,Лицей,Биология,8.0,да,4,5,8.0,8.0,2.2,57,да,1.142857
9,10,Михаил,М,17,Екатеринбург,Гимназия,Информатика,3.7,нет,9,5,8.0,8.1,3.3,72,да,0.528571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,София,Ж,17,Москва,Обычная,История,8.4,нет,5,6,2.0,7.1,2.6,59,да,1.200000
996,997,Дарья,Ж,17,Казань,Гимназия,Химия,4.3,да,1,4,5.0,NaN,2.9,66,да,0.614286
997,998,Матвей,М,17,Самара,Обычная,История,5.9,нет,4,7,2.0,6.9,2.4,48,да,0.842857
998,999,Алиса,Ж,18,Ростов-на-Дону,Обычная,Русский язык,3.1,нет,9,6,8.0,6.2,3.9,74,да,0.442857


> Есть удобная конвенция: если хочешь удалить **пропуски** — используешь `.dropna()`
> (мы разбирали её в прошлом занятии), если удалить **дубликаты** — `.drop_duplicates()`.
> Обычный `.drop()` — только когда точно знаешь, какие метки хочешь убрать.


---
## Часть 2. Свои функции и `.apply()`

Часто нужно применить к столбцу **не готовую операцию** (`+`, `-`, `.mean()`),
а **свою собственную логику**. Например: «поставить категорию по баллу», «взять первую
букву имени», «пересчитать возраст в месяцы». Для этого есть **`.apply()`** — метод,
который **применяет функцию к каждому элементу** столбца (или к каждой строке таблицы).

### 2.1. Простой пример: обычная функция

Напишем функцию, которая по баллу ЕГЭ выдаёт словесную категорию:

In [114]:
def category(x):
    if x >= 80:
        return 'высокий'
    elif x >= 60:
        return 'средний'
    else:
        return 'низкий'

# применим её к каждому значению столбца
df_work['категория'] = df_work['балл_егэ'].apply(category)
df_work[['имя', 'балл_егэ', 'категория']].head(10)

,имя,балл_егэ,категория
0,Роман,47,низкий
1,Арина,66,средний
2,Дарья,80,высокий
3,Владимир,58,низкий
4,Дмитрий,61,средний
5,Милана,81,высокий
6,Роман,90,высокий
7,Дарья,48,низкий
8,Тимофей,57,низкий
9,Михаил,72,средний


**Что произошло:** Pandas прошёлся по каждому значению столбца `балл_егэ`,
для каждого вызвал функцию `category(x)` и собрал результаты в новый столбец.

### 2.2. Анонимные функции — `lambda`

Если функция очень короткая — писать её отдельно через `def` не хочется. Тогда используют
**`lambda`** — «анонимную функцию», записанную одной строкой:

```python
lambda x: <что вернуть>
```

Это то же самое, что:
```python
def безымянная(x):
    return <что вернуть>
```

**Пример:** сделаем столбец «часов в день» через `lambda`:

In [115]:
# то же деление на 7, но через lambda прямо в apply
df_work['часов_в_день_v2'] = df_work['часов_неделя'].apply(lambda x: round(x / 7, 2))
df_work[['имя', 'часов_неделя', 'часов_в_день_v2']].head()

,имя,часов_неделя,часов_в_день_v2
0,Роман,5.4,0.77
1,Арина,9.1,1.30
2,Дарья,11.3,1.61
3,Владимир,6.2,0.89
4,Дмитрий,6.5,0.93


**Когда какой способ?**

| Функция | Когда удобно |
|---|---|
| Обычная `def имя(...)` | Логика **сложная**, много строк, нужны условия / несколько операций |
| **`lambda x: ...`** | Логика **простая**, помещается в одну строку |

`lambda` часто пишут прямо внутри `.apply()` — коротко и на месте.

### 2.3. `.apply()` по строкам целиком

Иногда нужно посчитать что-то **из нескольких столбцов сразу** — тогда применяют
функцию к **строке** таблицы, а не к отдельной ячейке. Для этого добавляют `axis=1`:

In [116]:
# «интегральный балл»: средний балл года × 20 + балл ЕГЭ / 2
df_work['итог'] = df_work.apply(
    lambda row: row['средний_балл_года'] * 20 + row['балл_егэ'] / 2,
    axis=1
).round(1)

df_work[['имя', 'средний_балл_года', 'балл_егэ', 'итог']].head()

,имя,средний_балл_года,балл_егэ,итог
0,Роман,2.4,47,71.5
1,Арина,3.4,66,101.0
2,Дарья,2.6,80,92.0
3,Владимир,2.6,58,81.0
4,Дмитрий,3.0,61,90.5


**Разница по `axis`:**
- `axis=0` (по умолчанию для `.apply()` на `DataFrame`) — функция получает **столбец целиком**;
- `axis=1` — функция получает **строку целиком** (в виде `Series`), и внутри можно
  обращаться к её значениям как к словарю: `row['имя_столбца']`.


---
## Часть 3. Быстрая работа со строковыми столбцами — `.str`

У каждого текстового столбца в Pandas есть особый «аксессор» **`.str`**, через который
можно применять строковые операции **сразу ко всей колонке** — без `.apply()` и без циклов.

### Примеры:

In [117]:
# все имена в верхний регистр
df_work['имя'].str.upper().head()

0       РОМАН
1       АРИНА
2       ДАРЬЯ
3    ВЛАДИМИР
4     ДМИТРИЙ
Name: имя, dtype: object

In [ ]:
# длина имени
df_work['имя'].str.len().head()

0    5
1    5
2    5
3    8
4    7
Name: имя, dtype: int64

In [119]:
# первая буква имени
df_work['имя'].str[0].head()

0    Р
1    А
2    Д
3    В
4    Д
Name: имя, dtype: object

In [120]:
# проверяем, содержит ли город слово «Санкт»
df_work['город'].str.contains('Санкт').head()

0    False
1    False
2    False
3    False
4    False
Name: город, dtype: bool

In [121]:
# заменим дефис в названии города на пробел
df_work['город'].str.replace('-', ' ').head()

0    Нижний Новгород
1     Ростов на Дону
2             Самара
3     Ростов на Дону
4             Москва
Name: город, dtype: object

**Полезные методы `.str`:**

| Метод | Что делает |
|---|---|
| `.str.upper()` / `.str.lower()` | верхний / нижний регистр |
| `.str.len()` | длина строки |
| `.str.strip()` | убрать пробелы по краям |
| `.str.contains('...')` | содержит ли подстроку (возвращает маску!) |
| `.str.startswith('...')` | начинается ли с подстроки |
| `.str.replace('a', 'b')` | заменить подстроку |
| `.str.split('...')` | разбить по разделителю |
| `.str[0]` | первый символ (как индексация строки) |

`.str.contains(...)` часто комбинируют с булевой маской — это удобный способ фильтрации
по текстовым столбцам:

In [122]:
# только ученики из городов, в названии которых есть «Санкт» или «Ново»
mask = df_work['город'].str.contains('Санкт|Ново')
df_work[mask]

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_неделя,репетитор,пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал,часов_в_день,категория,часов_в_день_v2,итог
8,9,Тимофей,М,16,Санкт-Петербург,Лицей,Биология,8.0,да,4,5,8.0,8.0,2.2,57,да,1.142857,низкий,1.14,72.5
10,11,Алиса,Ж,16,Санкт-Петербург,Обычная,Биология,7.6,нет,6,4,5.0,NaN,2.9,64,да,1.085714,средний,1.09,90.0
15,16,Ева,Ж,17,Новосибирск,Гимназия,Английский язык,8.8,нет,6,3,5.0,NaN,3.2,70,да,1.257143,средний,1.26,99.0
23,24,Ева,Ж,17,Новосибирск,Обычная,Биология,12.2,да,8,2,6.0,7.5,3.1,83,да,1.742857,высокий,1.74,103.5
24,25,Ева,Ж,17,Новосибирск,Обычная,Обществознание,14.6,да,6,2,7.0,10.1,3.9,100,да,2.085714,высокий,2.09,128.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,987,Ева,Ж,17,Новосибирск,Обычная,Физика,0.0,да,6,1,4.0,8.2,2.8,47,да,0.000000,низкий,0.00,79.5
987,988,Максим,М,16,Новосибирск,Гимназия,Математика,0.1,да,7,7,8.0,6.3,2.9,69,да,0.014286,средний,0.01,92.5
990,991,Михаил,М,17,Новосибирск,Обычная,Обществознание,4.8,нет,5,11,7.0,8.3,2.3,54,да,0.685714,низкий,0.69,73.0
992,993,Артём,М,17,Новосибирск,Гимназия,Математика,14.0,нет,3,9,NaN,6.8,2.4,58,да,2.000000,низкий,2.00,77.0


---
## Часть 4. `pd.concat()` — склеивание таблиц

Часто данные приходят **несколькими кусками**: файл за январь, за февраль, за март.
Их нужно соединить в одну большую таблицу. Для этого используют **`pd.concat()`**.

### 4.1. Склейка «стопкой» (по строкам)

Если таблицы имеют **одинаковые столбцы**, `concat` ставит их друг под другом:

In [123]:
# разобьём наш датасет на две половины — как будто это два файла
first = df.iloc[:500].copy()
second = df.iloc[500:].copy()

In [124]:
first

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,496,Егор,М,17,Новосибирск,Обычная,Русский язык,4.8,да,10,6,7.0,6.0,2.8,58,да
496,497,Михаил,М,17,Екатеринбург,Обычная,Математика,4.6,да,9,3,6.0,8.8,3.7,63,да
497,498,Тимофей,М,17,Самара,Обычная,Математика,8.2,нет,8,4,5.0,9.3,3.5,81,да
498,499,Ксения,Ж,16,Казань,Обычная,Информатика,5.0,да,6,5,9.0,6.9,2.6,74,да


In [125]:
second

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
500,501,Михаил,М,17,Ростов-на-Дону,Лицей,Химия,9.2,нет,8,6,4.0,8.6,2.6,46,да
501,502,Алиса,Ж,17,Самара,Гимназия,Физика,6.4,да,3,10,8.0,6.5,2.0,46,да
502,503,Алиса,Ж,17,Екатеринбург,Гимназия,История,2.8,нет,2,8,3.0,6.6,2.1,30,нет
503,504,Виктория,Ж,18,Санкт-Петербург,Обычная,Обществознание,6.5,да,16,7,5.0,7.9,3.4,76,да
504,505,Данил,М,17,Ростов-на-Дону,Лицей,Физика,9.6,да,5,2,7.0,6.3,3.5,87,да
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,София,Ж,17,Москва,Обычная,История,8.4,нет,5,6,2.0,7.1,2.6,59,да
996,997,Дарья,Ж,17,Казань,Гимназия,Химия,4.3,да,1,4,5.0,NaN,2.9,66,да
997,998,Матвей,М,17,Самара,Обычная,История,5.9,нет,4,7,2.0,6.9,2.4,48,да
998,999,Алиса,Ж,18,Ростов-на-Дону,Обычная,Русский язык,3.1,нет,9,6,8.0,6.2,3.9,74,да


In [126]:
# склеиваем обратно
combined = pd.concat([first, second])
print('После concat:', combined.shape)
combined.head()

После concat: (1000, 16)


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


**Важно:** у склеенной таблицы индексы могут «повториться» (0..499 + 500..999) —
иногда это ок, иногда нужен свежий индекс. Тогда добавляют `ignore_index=True`:

In [127]:
combined = pd.concat([first, second], ignore_index=True)
combined

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,София,Ж,17,Москва,Обычная,История,8.4,нет,5,6,2.0,7.1,2.6,59,да
996,997,Дарья,Ж,17,Казань,Гимназия,Химия,4.3,да,1,4,5.0,NaN,2.9,66,да
997,998,Матвей,М,17,Самара,Обычная,История,5.9,нет,4,7,2.0,6.9,2.4,48,да
998,999,Алиса,Ж,18,Ростов-на-Дону,Обычная,Русский язык,3.1,нет,9,6,8.0,6.2,3.9,74,да


### 4.2. Склейка «в ширину» (по столбцам)

Если хочется поставить таблицы **рядом**, а не одна под другой, — параметр `axis=1`.
Строки при этом должны быть одинаковыми (тот же индекс).

In [128]:
# возьмём два кусочка от одной таблицы — как будто у нас два источника про одних людей
left_part  = df[['имя', 'пол', 'возраст']].head()
right_part = df[['балл_егэ', 'сдал']].head()

In [129]:
left_part

,имя,пол,возраст
0,Роман,М,17
1,Арина,Ж,17
2,Дарья,Ж,17
3,Владимир,М,16
4,Дмитрий,М,17


In [130]:
right_part

,балл_егэ,сдал
0,47,да
1,66,да
2,80,да
3,58,да
4,61,да


In [131]:
pd.concat([left_part, right_part], axis=1)

,имя,пол,возраст,балл_егэ,сдал
0,Роман,М,17,47,да
1,Арина,Ж,17,66,да
2,Дарья,Ж,17,80,да
3,Владимир,М,16,58,да
4,Дмитрий,М,17,61,да


> **Когда `concat`?** Когда таблицы **однородные** — те же столбцы (для склейки по строкам)
> или те же строки (для склейки по столбцам). Если структура разная — нужен `merge`, о нём дальше.


---
## Часть 5. `pd.merge()` — соединение по ключу

Самый мощный инструмент — **`pd.merge()`**. Он соединяет две таблицы **по общему столбцу-ключу**,
как `JOIN` в SQL. Классический пример: в основной таблице есть колонка `город`, а в другой
таблице для каждого города записан **регион и население**. Хочется добавить эти сведения
к каждому ученику.

### 5.1. Готовим справочник городов

Сделаем маленькую справочную таблицу:

In [132]:
cities_info = pd.DataFrame({
    'город': ['Москва', 'Санкт-Петербург', 'Казань', 'Новосибирск',
              'Екатеринбург', 'Нижний Новгород', 'Самара', 'Ростов-на-Дону'],
    'регион':          ['ЦФО', 'СЗФО', 'ПФО', 'СФО', 'УФО', 'ПФО', 'ПФО', 'ЮФО'],
    'население_млн':   [13.1, 5.6, 1.3, 1.6, 1.5, 1.2, 1.1, 1.1],
})
cities_info

,город,регион,население_млн
0,Москва,ЦФО,13.1
1,Санкт-Петербург,СЗФО,5.6
2,Казань,ПФО,1.3
3,Новосибирск,СФО,1.6
4,Екатеринбург,УФО,1.5
5,Нижний Новгород,ПФО,1.2
6,Самара,ПФО,1.1
7,Ростов-на-Дону,ЮФО,1.1


### 5.2. Соединяем таблицы через `merge`

In [133]:
df_merged = df.merge(cities_info, on='город', how='left')
df_merged[['имя', 'город', 'регион', 'население_млн']].head()

,имя,город,регион,население_млн
0,Роман,Нижний Новгород,ПФО,1.2
1,Арина,Ростов-на-Дону,ЮФО,1.1
2,Дарья,Самара,ПФО,1.1
3,Владимир,Ростов-на-Дону,ЮФО,1.1
4,Дмитрий,Москва,ЦФО,13.1


**Что произошло:** Pandas для каждой строки таблицы `df` посмотрел на её `город`,
нашёл этот город в справочнике `cities_info` и **подставил** оттуда `регион` и `население_млн`.
Теперь к каждому ученику привязаны ещё два признака — их можно использовать в анализе.

### 5.3. Типы соединений: `how=`

Ключевой параметр `merge` — это **`how`**. Он говорит, **какие строки сохранять**,
если в одной таблице ключ есть, а в другой — нет.

| Тип | Что оставит |
|---|---|
| **`inner`** (по умолчанию) | Только те строки, где **ключ есть в ОБЕИХ** таблицах |
| **`left`** | **Все строки левой** таблицы (правые данные подставляются, если нашлись) |
| **`right`** | **Все строки правой** таблицы (симметрично `left`) |
| **`outer`** | **Все строки из обеих** таблиц (пропуски заполнятся `NaN`) |

Наглядно все четыре варианта через **круги Эйлера**:

![Типы соединений SQL/merge](images/joins_euler.jpg)

Слева всегда — левая таблица, справа — правая. Закрашенная область — те строки,
которые попадут в результат.

**Иллюстрация на маленьком примере.** Сделаем две таблички: в левой — имена (id 1–4),
в правой — города (id 3–6). Общие id — **3 и 4**.

In [134]:
left = pd.DataFrame({
    'id':   [1, 2, 3, 4],
    'имя':  ['Аня', 'Боря', 'Вика', 'Гена'],
})
right = pd.DataFrame({
    'id':    [3, 4, 5, 6],
    'город': ['Москва', 'Казань', 'Самара', 'Уфа'],
})
print('LEFT:'); print(left)
print('\nRIGHT:'); print(right)

LEFT:
   id   имя
0   1   Аня
1   2  Боря
2   3  Вика
3   4  Гена

RIGHT:
   id   город
0   3  Москва
1   4  Казань
2   5  Самара
3   6     Уфа


#### `inner` — только общие ключи

Остаются только те строки, у которых `id` **есть в обеих таблицах** — то есть 3 и 4.
Остальные строки (id 1, 2, 5, 6) отбрасываются.

![inner join](images/merge_inner.png)

In [135]:
# inner — только id, которые есть В ОБЕИХ таблицах (3 и 4)
left.merge(right, on='id', how='inner')

,id,имя,город
0,3,Вика,Москва
1,4,Гена,Казань


#### `left` — все строки левой таблицы

Берём **все имена** (id 1–4). Где в правой таблице нашёлся город — подставляем его (id 3, 4),
где не нашёлся (id 1, 2) — пишем `NaN`. Строки из правой таблицы, которых нет слева (id 5, 6),
теряются.

![left join](images/merge_left.png)

In [136]:
# left — все имена, город подставится только для id 3 и 4
left.merge(right, on='id', how='left')

,id,имя,город
0,1,Аня,NaN
1,2,Боря,NaN
2,3,Вика,Москва
3,4,Гена,Казань


#### `right` — все строки правой таблицы

Зеркально `left`: берём **все города** (id 3–6), а имя подставляется только для 3 и 4 — остальное `NaN`.
Строки из левой, которых нет справа (id 1, 2), теряются.

![right join](images/merge_right.png)

In [137]:
# right — все города, имя подставится только для id 3 и 4
left.merge(right, on='id', how='right')

,id,имя,город
0,3,Вика,Москва
1,4,Гена,Казань
2,5,NaN,Самара
3,6,NaN,Уфа


#### `outer` — все ключи из обеих таблиц

Не теряем ничего: в результате будут все id (1–6). Где в одной из таблиц данных нет — ставим `NaN`.

![outer join](images/merge_outer.png)

In [138]:
# outer — вообще все id из обеих таблиц
left.merge(right, on='id', how='outer')

,id,имя,город
0,1,Аня,NaN
1,2,Боря,NaN
2,3,Вика,Москва
3,4,Гена,Казань
4,5,NaN,Самара
5,6,NaN,Уфа


### 5.4. Что если ключи называются по-разному?

До сих пор в обеих таблицах ключевой столбец назывался одинаково — `город`. Тогда
хватало `on='город'`. Но часто на практике **имена не совпадают**: в основной таблице —
`город`, а в справочнике — `city`. Переименовывать необязательно — можно указать
**два ключа** через `left_on=` и `right_on=`.


In [ ]:
students = pd.DataFrame({
    'имя':   ['Аня', 'Боря', 'Вика', 'Гена'],
    'город': ['Москва', 'Казань', 'Уфа', 'Москва'],
})

cities_info = pd.DataFrame({
    'city':   ['Москва', 'Казань', 'Уфа', 'Пермь'],
    'регион': ['Центр',  'Поволжье', 'Урал', 'Урал'],
})

students.merge(cities_info, left_on='город', right_on='city', how='left')


**Что произошло:** Pandas взял значение из столбца `город` слева, нашёл такое же в столбце
`city` справа и подтянул `регион`. Оба столбца-ключа при этом **остаются** в результате
(и `город`, и `city`) — один из них можно сразу удалить через `.drop(columns='city')`,
если он не нужен.

> Если ключевые столбцы называются **одинаково** — короче писать просто `on='...'`.


### 5.5. Соединение сразу по нескольким столбцам

Иногда одного ключа **не хватает**, чтобы однозначно определить строку. Классический
пример: у одного ученика есть оценки по разным предметам, и учитель для «Ани по математике»
и «Ани по русскому» — **разный**. Тогда ключ — это **пара** `(имя, предмет)`.

В `merge` можно передать **список столбцов** — Pandas будет искать совпадение сразу по всем.


In [ ]:
scores = pd.DataFrame({
    'имя':     ['Аня', 'Аня', 'Боря', 'Боря', 'Вика'],
    'предмет': ['матем', 'рус', 'матем', 'рус', 'матем'],
    'балл':    [82, 74, 60, 55, 91],
})

teachers = pd.DataFrame({
    'имя':     ['Аня', 'Аня', 'Боря', 'Боря'],
    'предмет': ['матем', 'рус', 'матем', 'рус'],
    'учитель': ['Иванов', 'Петров', 'Иванов', 'Сидоров'],
})

scores.merge(teachers, on=['имя', 'предмет'], how='left')


**Что произошло:** Pandas для каждой строки `scores` искал в `teachers` строку с **точно
такой же парой** `(имя, предмет)`. Для Ани по математике нашёл Иванова, для Ани по русскому —
Петрова, хотя имя то же самое. Для Вики по математике совпадения не нашлось — в результате `NaN`.

> Если бы мы соединили только по `имя`, для «Ани по математике» Pandas нашёл бы **обе** её
> строки в `teachers` — и результат бы «раздулся» с дубликатами. Правильно выбранный набор
> ключей — это то, что делает `merge` **однозначным**.


### 5.6. Когда что использовать?

- **`left`** — самый частый выбор в анализе. Мы **не хотим терять строки** основной
  таблицы, а из справочника хотим просто **дополнить** данные.
- **`inner`** — когда нужны только совпадающие строки (например, для расчёта пересечений).
- **`right`** — редко, обычно проще поменять таблицы местами и написать `left`.
- **`outer`** — когда нужно **не потерять ничего**: например, объединить два списка клиентов
  и посмотреть, кто есть только у одного из вас, кто есть у обоих.



---
## Итог занятия

Сегодня научились:
- **менять структуру таблицы** — добавлять столбцы через `df['новый'] = ...`,
  удалять через `.drop(..., axis=1)`, переименовывать через `.rename(columns={...})`;
- применять **свои функции** через **`.apply()`** — по столбцу или по строке (`axis=1`);
- использовать **`lambda`** для коротких функций прямо на месте;
- быстро обрабатывать текстовые столбцы через **`.str.upper()`, `.str.contains(...)`** и другие;
- **склеивать таблицы** через `pd.concat()` (когда столбцы одинаковые);
- **соединять таблицы по ключу** через `pd.merge()`: 4 типа соединения (`inner`, `left`, `right`, `outer`), а также `left_on=` / `right_on=` для ключей с разными именами и `on=[col1, col2]` для соединения сразу по нескольким столбцам.

**Главная мысль:** реальный анализ почти всегда — это **не одна таблица**, а **несколько**,
которые нужно **преобразовать**, **склеить** и **обогатить** справочными данными.
